In [ ]:
import os
import csv

import numpy as np
import xarray as xr
import scipy.io as sio

import matplotlib.pyplot as plt

# settings
%config InlineBackend.figure_format = 'retina'

# All data this notebook reads is repo-relative: data/processed/ for the MATLAB pipeline's
# output and data/external/ for the LR04 stack. So there is no PROXY_DATA_DIR here -- that
# variable pointed at the raw proxy tree, which only the MATLAB stage reads. Launch Jupyter
# from the repo root and the paths below resolve with no setup at all.
extern = 'data/external'
# save figs here
opath = os.environ.get('FIG_OUTPUT_DIR')
if not opath:
    raise RuntimeError(
        "FIG_OUTPUT_DIR is not set. Run `source config/paths.env` before launching Jupyter "
        "(see config/paths.env.example). Refusing to guess a default: the old fallback wrote to "
        "a repo-local outputs/ that tools/sync_manuscript_figs.sh never read, and figures "
        "silently diverged between the two directories."
    )
os.makedirs(opath, exist_ok=True)

In [ ]:
# --- LOAD PROXY DATA --- #

# Read straight from the MATLAB pipeline's output.
# The scripts/matlab/dDwax_data_processing_{nh22p,d480_d479}.m write the *_FAMEs_current.mat files
# that carry every field this notebook needs: age, dDraw, stdev, dDivc, dDp, jas
#
# *_FAMEs_current.mat is deliberately undated: the MATLAB saves both a dated archive copy and
# this stable name, so nothing here has to name a date and quietly go stale

def load_fames(path, key):
    """Load a *_FAMEs_current.mat struct as the xr.Dataset the rest of this notebook expects.

    MATLAB structs come out of loadmat as nested object arrays; simplify_cells=True unwraps
    them to a plain dict of arrays and squeezes the (n,1) columns to (n,). The dDp and jas
    ensembles keep their own ensemble dimensions because they differ in width -- dDp is 1000
    Monte Carlo draws, jas is 4000 (10 Gibbs chains x 800, thinned by 2).
    """
    s = sio.loadmat(path, simplify_cells=True)[key]
    ds = xr.Dataset(
        {v: ('age', s[v]) for v in ('dDraw', 'stdev', 'dDivc')},
        coords={'age': s['age']},
    )
    # 'jas' is the MATLAB field name; 'pJAS' is what this notebook calls it
    for name, mat in (('dDp', s['dDp']), ('pJAS', s['jas'])):
        dim = f'ensemble_n_{name}'
        ds = ds.assign_coords({dim: np.arange(mat.shape[1])})
        ds[name] = (('age', dim), mat)
    return ds

procd = 'data/processed'

# nh22p
nh22p = load_fames(f'{procd}/nh22p_FAMEs_current.mat', 'nh22p')

# dsdp-480-479 
# the saved struct is already the 147-sample d480+d479 composite, spliced and
# age-sorted in MATLAB (dDwax_data_processing_d480_d479.m), not assembled here
d480_479 = load_fames(f'{procd}/Guaymas_d480_d479_FAMEs_current.mat', 'Guay')

# lr04
# Read from data/external/lr04.mat: `delob` columns are [age (ka), d18O (per mil), error]
delob = sio.loadmat(f'{extern}/lr04.mat')['delob']
lr04_age = delob[:, 0]
lr04_d18O = delob[:, 1]

In [ ]:
# --- INTERVAL DIFFS FOR dDC30 TIMESERIES --- #

cores     = ['d480_479','nh22p']
intervals = ['late-holocene','holocene','lgm','lig','pgm']

# init dictionaries
age_bnds        = { interval: {} for interval in intervals }
dDtimeslice     = { core: { interval: {} for interval in intervals } for core in cores }
interval_mean   = { core: { interval: {} for interval in intervals } for core in cores }
interval_stddev = { core: { interval: {} for interval in intervals } for core in cores }
differences     = { core: { modern_i: { i: {} for i in ['lgm','lig','pgm'] } for modern_i in ['late-holocene','holocene'] } for core in cores }

# populate age bounds (ka)
age_bnds['late-holocene'] = [0, 4]
age_bnds['holocene']      = [0, 11.7]
age_bnds['lgm']           = [18, 24]
age_bnds['lig']           = [117, 130]
age_bnds['pgm']           = [135, 150]

for core in cores:
    print(f'\n{core}')
    for interval in intervals:
        # subset dD wax time-series
        t_min, t_max = age_bnds[interval][0], age_bnds[interval][1]
        if core=='d480_479':
            timeslice_subset = d480_479.sel(age=slice(t_min,t_max))
        elif core=='nh22p':
            timeslice_subset = nh22p.sel(age=slice(t_min,t_max))
        dDtimeslice[core][interval] = timeslice_subset
        # calculate timeslice stats
        interval_mean[core][interval]=dDtimeslice[core][interval].dDraw.mean(dim=['age'])
        interval_stddev[core][interval]=dDtimeslice[core][interval].dDraw.std(dim=['age'])
        print(f'{interval} mean, std dev = {np.round(interval_mean[core][interval],2).values}, {np.round(interval_stddev[core][interval],2).values}')

# print interval differences
for core in cores:
    print(f'\n{core}')
    for modern_ref in ['late-holocene','holocene']:
        for interval in ['lgm','lig','pgm']:
            differences[core][modern_ref][interval] = np.round(interval_mean[core][interval].values - interval_mean[core][modern_ref].values,2)
            print(f'{interval} - {modern_ref} = {differences[core][modern_ref][interval]}')

In [ ]:
### create data frame for exporting
# Writes to data/processed/
# Path is relative to the repo root, which is where notebooks must be launched from.
# These CSVs are tracked in git — they are how the Casper clone receives proxy data.
#
# Header and rows are built from `cores` and `intervals` rather than typed out, so adding a
# timeslice in the cell above flows through to the CSV without editing a literal header.
#
# Column suffix is `_stddev`, not the old `_1serr`: the quantity is the standard deviation of
# the dDraw samples falling inside each age window (`.std(dim=['age'])` above), i.e. the spread
# of the data, not the standard error of the window mean. The old name overstated the precision
# of the mean by a factor of sqrt(n).
#
# The two Holocene windows are both exported and are NOT interchangeable:
#   late_holocene  0-4 ka     — what this CSV has always carried, and what the model notebooks
#                               difference against; keeping it under its own name is why those
#                               numbers are unchanged by the rename.
#   holocene       0-11.7 ka  — the wider interglacial window fig3 shades as a band.
# See data/processed/README.md; CLAUDE.md records that these windows are deliberately not unified.

core_meta = {                        # dict key -> (csv core_name, lon, lat)
    'd480_479': ('DSDP_480_479', -111.62,   27.85),
    'nh22p':    ('NH22P',        -106.5183, 22.5183),
}

header = ['core_name', 'lon', 'lat']
for interval in intervals:
    col = interval.replace('-', '_')          # 'late-holocene' -> 'late_holocene'
    header += [f'{col}_dD', f'{col}_dD_stddev']

data = [header]
for core in cores:
    core_name, lon, lat = core_meta[core]
    row = [core_name, f'{lon}', f'{lat}']
    for interval in intervals:
        row += [f"{float(interval_mean[core][interval].values)}",
                f"{float(interval_stddev[core][interval].values)}"]
    data.append(row)

with open('data/processed/timeslice_mean_proxy_dDraw.csv', "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(data)


In [ ]:
# --- MIS stage definitions and band drawing, shared by both timeseries figures ---------------

COLD_MIS        = np.array([[14,29],[38,45],[57,71],[84,95],[105,114],[135,141],[141,150]])
COLD_MIS_LABELS = ['2', '3b', '4', '5b', '5d', '6a', '6b']
WARM_MIS        = np.array([[0,14],[29,38],[45,57],[71,84],[95,105],[114,135]])
WARM_MIS_LABELS = ['1', '3a', '3c', '5a', '5c', '5e']
INTGLCL         = np.array([[0,11.7],[117,130]])
INTGLCL_LABELS  = ['HOL', 'LIG']


def mis_bands(ages, y0, y1):
    """(n,2) stage ages -> the (n,4) [start, end, y_bottom, y_top] array the drawing uses."""
    n = len(ages)
    return np.column_stack([ages, np.full(n, y0), np.full(n, y1)])


def band_xpos(ages):
    """Mid-age of each stage, for placing its label."""
    return np.mean(ages, axis=1)


def draw_stage_bands(ax, bands, patch_kw, labels=None, label_xpos=None, label_y=None,
                     label_kw=None):
    """Shade one set of stage bands on ax, optionally labelling each.

    bands : (n,4) array of [start_ka, end_ka, y_bottom, y_top], i.e. the output of mis_bands().
    Drawn at zorder 0 and hidden from the legend, matching the original inline loops.
    """
    for i in range(len(bands)):
        ax.add_patch(plt.Rectangle(
            (bands[i,0], bands[i,2]),               # lower-left corner
            bands[i,1] - bands[i,0],                # width
            bands[i,3] - bands[i,2],                # height
            zorder=0, label='_Hidden', **patch_kw))
        if labels is not None:
            ax.text(label_xpos[i], label_y, labels[i], **label_kw)


## dDraw timeseries

In [ ]:
cold_mis_boundaries = mis_bands(COLD_MIS, -134, -131)
warm_mis_boundaries = mis_bands(WARM_MIS, -134, -131)
intglcl_boundaries  = mis_bands(INTGLCL, -200, 0)
cold_mis_labels, warm_mis_labels, intglcl_labels = COLD_MIS_LABELS, WARM_MIS_LABELS, INTGLCL_LABELS
coldlabel_xpos, warmlabel_xpos, intglcl_xpos = band_xpos(COLD_MIS), band_xpos(WARM_MIS), band_xpos(INTGLCL)


In [ ]:
line_kw={'ls':'-', 'lw':2} #, 'marker':'s', 'mec':'k', 'mew':0.25} 
patch_kw = {'ec':'indianred', 'lw':0.5, 'linestyle':'-', 'fc':'indianred', 'alpha':0.25}
patch_kw2 = {'ec':'k', 'lw':0.5, 'linestyle':'-', 'fc':'white', 'alpha':1, 'clip_on':False}
patch_kw3 = {'ec':'k', 'lw':0.5, 'linestyle':'-', 'fc':'silver', 'alpha':1, 'clip_on':False}
tkw = {'axis':'both', 'direction':'out', 'labelsize': 11}
title_text_kw={'size':15, 'weight':'bold', 'color':'k', 'va':'center'}
label_text_kw={'size':12, 'weight':'bold', 'color':'firebrick', 'ha':'center', 'va':'bottom'} #'backgroundcolor':'white', 
label_text_kw2={'size':9, 'weight':'bold', 'color':'k', 'ha':'center', 'va':'center'}
laxis_text_kw={'weight':'normal', 'rotation':90, 'size':11, 'color':'k'}
raxis_text_kw={'weight':'normal', 'rotation':270, 'size':11, 'color':'grey'}
legend_kw = {'loc':1, 'fontsize':8, 'labelcolor':'linecolor', 'frameon':False}
# colors
sig1 = np.array([255, 196, 0]) / 255
sig2 = np.array([255, 242, 156]) / 255
colline = np.array([235, 142, 5]) / 255
# plot specs
xmin=0
xmax=150


## ++ Make Fig ++ ##
#fig, axs = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
fig = plt.figure(figsize=(9,6))
gs = fig.add_gridspec(2, hspace=0)
axs = gs.subplots(sharex=True, sharey=False)

## DSDP-480/479
# proxy data
ax1=axs[0]
ymin=-163 ; ymax=-134
ax1.text(1.5, ymax-2.5, 'DSDP-480/479', **title_text_kw)

# dDraw
ax1.fill_between(d480_479.age, d480_479.dDraw-d480_479.stdev, d480_479.dDraw+d480_479.stdev,
                 color=sig1, edgecolor='none', alpha=0.75, label='error')
ax1.plot(d480_479.age, d480_479.dDraw, color=colline, linewidth=1.5, label='mean')
ax1.set(xlim=[xmin,xmax], ylim=[ymin,ymax], yticks=np.arange(-163,-130,10))
ax1.set_ylabel(u'$\delta$D$_{C30}$ [‰]', labelpad=1, **laxis_text_kw)
ax1.tick_params(color='k', labelcolor='k', top=False, bottom=False, **tkw)
ax1.minorticks_on()
ax1.spines['left'].set_color('k')
ax1.spines['right'].set_color('none')
#ax1.spines['bottom'].set_color('none')

# benthic stack
ax2=ax1.twinx() 
ax2.plot(lr04_age, lr04_d18O, c='grey', **line_kw, label='LR04') # plot lr04 d18O data
ax2.set(xlabel='AGE (ka)', xlim=[xmin,xmax], ylim=[5.05,3.05], yticks=[5.0,4.5,4.0,3.5])
ax2.set_ylabel(u'LR04 $\delta^{18}O_{benthic}$ [‰]', labelpad=17, **raxis_text_kw)
ax2.tick_params(color='grey', labelcolor='grey', top=False, bottom=False, **tkw)
ax2.spines['right'].set_color('grey')
#ax2.spines['bottom'].set_color('none')

draw_stage_bands(ax1, warm_mis_boundaries, patch_kw2,
                 labels=warm_mis_labels, label_xpos=warmlabel_xpos,
                 label_y=ymax+1.35, label_kw=label_text_kw2)
    
draw_stage_bands(ax1, cold_mis_boundaries, patch_kw3,
                 labels=cold_mis_labels, label_xpos=coldlabel_xpos,
                 label_y=ymax+1.35, label_kw=label_text_kw2)
    
draw_stage_bands(ax1, intglcl_boundaries, patch_kw)

# re-order axes so proxy data is on top
ax1.set_zorder(ax2.get_zorder() + 1)
ax1.patch.set_visible(False)
#ax2.patch.set_visible(False)


## NH22P
# proxy data
ax3=axs[1]
ymin=-163 ; ymax=-134
ax3.text(1.5, ymax-2.5, 'NH22P', **title_text_kw)

# dDraw
ax3.fill_between(nh22p.age, nh22p.dDraw-nh22p.stdev, nh22p.dDraw+nh22p.stdev,
                 color=sig1, edgecolor='none', alpha=0.75, label='error')
ax3.plot(nh22p.age, nh22p.dDraw, color=colline, linewidth=1.5, label='mean') # mean time-series
ax3.set(xlim=[xmin,xmax], ylim=[ymin,ymax], yticks=np.arange(-163,-130,10))
ax3.set_ylabel(u'$\delta$D$_{C30}$ [‰]', labelpad=1, **laxis_text_kw)
ax3.tick_params(color='k', labelcolor='k', top=False, **tkw)
ax3.minorticks_on()
ax3.spines['left'].set_color('k')
ax3.spines['right'].set_color('none')
#ax3.spines['top'].set_color('none')
ax3.set_xlabel('AGE [ka]', color='k', weight='normal', size=11, rotation=0, labelpad=1)

# benthic stack
ax4=ax3.twinx() 
ax4.plot(lr04_age, lr04_d18O, c='grey', **line_kw, label='LR04') # plot lr04 d18O data
ax4.set(xlabel='AGE (ka)', xlim=[xmin,xmax], ylim=[5.05,3.05], yticks=[5.0,4.5,4.0,3.5])
ax4.set_ylabel(u'LR04 $\delta^{18}O_{benthic}$ [‰]', labelpad=17, **raxis_text_kw)
ax4.tick_params(color='grey', labelcolor='grey', top=False, **tkw)
ax4.spines['right'].set_color('grey')
ax4.spines['left'].set_color('none')

draw_stage_bands(ax3, intglcl_boundaries, patch_kw,
                 labels=intglcl_labels, label_xpos=intglcl_xpos,
                 label_y=ymin+0.5, label_kw=label_text_kw)
    
# re-order axes so proxy data is on top
ax3.set_zorder(ax4.get_zorder() + 1)
ax3.patch.set_visible(False)

#fig.text(0.075,-.1, '''Time-series of $\delta$D$_{wax}$-inferred $\delta$D$_{prec}$.\n
#DSDP-480/479 is based on Dervla handpicked values while NH22P is based on autopick method.\n
#Two white markers on left axis indicate mean JAS and JFM $\delta$D$_{prec}$ from OIPC, weighted by IMERG precip.''')
        
#ax[1].legend(**legend_kw)

plt.savefig(f'{opath}/dDivc_timeseries.pdf', bbox_inches='tight')

## dDp timeseries

In [ ]:
# Percentile bands are computed with np.nanpercentile directly in the plotting cell below.
# The previous version hardcoded `iters = 1000` and indexed into a sorted ensemble, which is
# only correct while both dDp sheets happen to be exactly 1000 members wide — that assumption
# silently broke once before, when the NH22P sheet was 1020 wide and the 97.5th-percentile band
# was drawn too narrow. See DATA_MANIFEST.md section 1b.

cold_mis_boundaries = mis_bands(COLD_MIS, -38, -35)
warm_mis_boundaries = mis_bands(WARM_MIS, -38, -35)
intglcl_boundaries  = mis_bands(INTGLCL, -100, 0)
cold_mis_labels, warm_mis_labels, intglcl_labels = COLD_MIS_LABELS, WARM_MIS_LABELS, INTGLCL_LABELS
coldlabel_xpos, warmlabel_xpos, intglcl_xpos = band_xpos(COLD_MIS), band_xpos(WARM_MIS), band_xpos(INTGLCL)


In [ ]:
line_kw={'ls':'-', 'lw':2} #, 'marker':'s', 'mec':'k', 'mew':0.25} 
patch_kw = {'ec':'indianred', 'lw':0.5, 'linestyle':'-', 'fc':'indianred', 'alpha':0.25}
patch_kw2 = {'ec':'k', 'lw':0.5, 'linestyle':'-', 'fc':'white', 'alpha':1, 'clip_on':False}
patch_kw3 = {'ec':'k', 'lw':0.5, 'linestyle':'-', 'fc':'silver', 'alpha':1, 'clip_on':False}
scat_kw = {'s': 90,'c': 'w', 'marker': 'o', 'edgecolors':'k', 'alpha':1, 'zorder': 100, 'clip_on': False}
tkw = {'axis':'y', 'direction':'out', 'labelsize': 10}
title_text_kw={'size':15, 'weight':'bold', 'color':'k', 'va':'center'}
label_text_kw={'size':12, 'weight':'bold', 'color':'firebrick', 'ha':'center', 'va':'bottom'} #'backgroundcolor':'white', 
laxis_text_kw={'weight':'normal', 'rotation':90, 'size':11, 'color':'k'}
raxis_text_kw={'weight':'normal', 'rotation':270, 'size':11, 'color':'grey'}
legend_kw = {'loc':1, 'fontsize':8, 'labelcolor':'linecolor', 'frameon':False}
# colors
sig1 = np.array([255, 196, 0]) / 255
sig2 = np.array([255, 242, 156]) / 255
colline = np.array([235, 142, 5]) / 255
# plot specs
xmin=0
xmax=150


## ++ Make Fig ++ ##
#fig, axs = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
fig = plt.figure(figsize=(9,6))
gs = fig.add_gridspec(2, hspace=0)
axs = gs.subplots(sharex=True, sharey=False)

## DSDP-480/479
# proxy data
ax1=axs[0]
ymin=-75; ymax=-38
ax1.text(1.5, -41, 'DSDP-480/479', **title_text_kw)
# bands as percentiles over the ensemble axis -- correct at any ensemble width
p480_479 = np.nanpercentile(d480_479['dDp'], [2.5, 16, 84, 97.5], axis=1) # -> (4, n_age)
ax1.fill_between(d480_479.age, p480_479[0], p480_479[3],
                 color=sig2, edgecolor='none', alpha=0.75, label='2$\sigma$') # 2-sigma shading
ax1.fill_between(d480_479.age, p480_479[1], p480_479[2],
                 color=sig1, edgecolor='none', alpha=0.75, label='1$\sigma$') # 1-sigma shading
ax1.plot(d480_479.age, np.nanmedian(d480_479['dDp'], axis=1), color=colline, linewidth=1.5, label='mean') # mean time-series
#ax1.scatter(0, Guaymas['jfm'], label='modern', **scat_kw) 
#ax1.scatter(0, Guaymas['jjas'], label='modern', **scat_kw) 
ax1.set(xlim=[xmin,xmax], ylim=[ymin,ymax], yticks=np.arange(-70,-35,5))
ax1.set_ylabel(u'$\delta$D$_{prec}$ [‰]', labelpad=1, **laxis_text_kw)
ax1.tick_params(color='k', labelcolor='k', top=False, bottom=False, **tkw)
ax1.minorticks_on()
ax1.spines['left'].set_color('k')
ax1.spines['right'].set_color('none')
#ax1.spines['bottom'].set_color('none')

# benthic stack
ax2=ax1.twinx() 
ax2.plot(lr04_age, lr04_d18O, c='grey', **line_kw, label='LR04') # plot lr04 d18O data
ax2.set(xlabel='AGE (ka)', xlim=[xmin,xmax], ylim=[5.05,3.05], yticks=[5.0,4.5,4.0,3.5])
ax2.set_ylabel(u'LR04 $\delta^{18}O_{benthic}$ [‰]', labelpad=17, **raxis_text_kw)
ax2.tick_params(color='grey', labelcolor='grey', top=False, bottom=False, **tkw)
ax2.spines['right'].set_color('grey')
#ax2.spines['bottom'].set_color('none')

draw_stage_bands(ax1, warm_mis_boundaries, patch_kw2,
                 labels=warm_mis_labels, label_xpos=warmlabel_xpos,
                 label_y=ymax+1.35, label_kw=label_text_kw2)
    
draw_stage_bands(ax1, cold_mis_boundaries, patch_kw3,
                 labels=cold_mis_labels, label_xpos=coldlabel_xpos,
                 label_y=ymax+1.35, label_kw=label_text_kw2)
    
draw_stage_bands(ax1, intglcl_boundaries, patch_kw)

# re-order axes so proxy data is on top
ax1.set_zorder(ax2.get_zorder() + 1)
ax1.patch.set_visible(False)
#ax2.patch.set_visible(False)


## NH22P
# proxy data
ax3=axs[1]
ymin=-73; ymax=-38
ax3.text(1.5, -41, 'NH22P', **title_text_kw)
p22p = np.nanpercentile(nh22p['dDp'], [2.5, 16, 84, 97.5], axis=1) # -> (4, n_age)
ax3.fill_between(nh22p.age, p22p[0], p22p[3], 
                color=sig2, edgecolor='none', alpha=0.75, label='2$\sigma$') # 2-sigma shading
ax3.fill_between(nh22p.age, p22p[1], p22p[2], 
                color=sig1, edgecolor='none', alpha=0.75, label='1$\sigma$') # 1-sigma shading
ax3.plot(nh22p.age, np.nanmedian(nh22p['dDp'], axis=1), color=colline, linewidth=1.5, label='mean') # mean time-series
#ax3.scatter(0, Mazatlan['jfm'], label='modern', **scat_kw) 
#ax3.scatter(0, Mazatlan['jjas'], label='modern', **scat_kw) 
ax3.set(xlim=[xmin,xmax], ylim=[ymin,ymax], yticks=np.arange(-70,-35,5))
ax3.set_ylabel(u'$\delta$D$_{prec}$ [‰]', labelpad=1, **laxis_text_kw)
ax3.tick_params(color='k', labelcolor='k', top=False, **tkw)
ax3.minorticks_on()
ax3.spines['left'].set_color('k')
ax3.spines['right'].set_color('none')
#ax3.spines['top'].set_color('none')
ax3.set_xlabel('AGE [ka]', color='k', weight='normal', size=11, rotation=0, labelpad=1)

# benthic stack
ax4=ax3.twinx() 
ax4.plot(lr04_age, lr04_d18O, c='grey', **line_kw, label='LR04') # plot lr04 d18O data
ax4.set(xlabel='AGE (ka)', xlim=[xmin,xmax], ylim=[5.05,3.05], yticks=[5.0,4.5,4.0,3.5])
ax4.set_ylabel(u'LR04 $\delta^{18}O_{benthic}$ [‰]', labelpad=17, **raxis_text_kw)
ax4.tick_params(color='grey', labelcolor='grey', top=False, **tkw)
ax4.spines['right'].set_color('grey')
ax4.spines['left'].set_color('none')

draw_stage_bands(ax3, intglcl_boundaries, patch_kw,
                 labels=intglcl_labels, label_xpos=intglcl_xpos,
                 label_y=ymin+0.5, label_kw=label_text_kw)
    
# re-order axes so proxy data is on top
ax3.set_zorder(ax4.get_zorder() + 1)
ax3.patch.set_visible(False)

#fig.text(0.075,-.1, '''Time-series of $\delta$D$_{wax}$-inferred $\delta$D$_{prec}$.\n
#DSDP-480/479 is based on Dervla handpicked values while NH22P is based on autopick method.\n
#Two white markers on left axis indicate mean JAS and JFM $\delta$D$_{prec}$ from OIPC.''')
        
#ax[1].legend(**legend_kw)

plt.savefig(f'{opath}/dDp_timeseries.pdf', bbox_inches='tight')